In [1]:
import pandas as pd
import numpy as np

print("1. Initializing Macro-Economic Data Wrangling Pipeline...")

# ==========================================
# 1. DATA INGESTION: FISCAL & DEMOGRAPHIC
# ==========================================
# Data extracted from:
# 1. BudgIT 2022 State of States Report (Fiscal Allocations)
# 2. National Bureau of Statistics (State Population Projections)

macro_data = {
    "State": [
        "ABIA", "ADAMAWA", "AKWA IBOM", "ANAMBRA", "BAUCHI", "BAYELSA", 
        "BENUE", "BORNO", "CROSS RIVER", "DELTA", "EBONYI", "EDO", 
        "EKITI", "ENUGU", "FCT ABUJA", "GOMBE", "IMO", "JIGAWA", 
        "KADUNA", "KANO", "KATSINA", "KEBBI", "KOGI", "KWARA", 
        "LAGOS", "NASARAWA", "NIGER", "OGUN", "ONDO", "OSUN", 
        "OYO", "PLATEAU", "RIVERS", "SOKOTO", "TARABA", "YOBE", "ZAMFARA"
    ],
    # Note: Replace placeholders with exact 'Public Order and Safety' figures for final publication
    "Security_Budget_Allocation": [
        0, 0, 153141288000, 57578248336, 57636322172, 79434163345, 
        0, 56612570572, 81028876273, 127784922685, 58092704789, 79021356471, 
        0, 0, 0, 0, 0, 59192951673, 
        162634878000, 73455617000, 57273260095, 0, 48444145297, 1800000000, 
        425606429000, 0, 0, 82380637191, 0, 0, 
        61161387619, 0, 413125225758, 62379304199, 0, 44420842619, 0
    ],
    "State_Population": [
        4260000, 4900000, 6500000, 6360000, 7740000, 2550000, 
        6800000, 6950000, 4470000, 6650000, 3310000, 4850000, 
        3820000, 5130000, 3560000, 3820000, 6350000, 6730000, 
        9590000, 15420000, 9100000, 5190000, 4970000, 3590000, 
        14730000, 2860000, 6510000, 6150000, 5430000, 5520000, 
        9290000, 4700000, 8660000, 5810000, 3590000, 3830000, 5300000 
    ]
}

macro_df = pd.DataFrame(macro_data)

# ==========================================
# 2. FEATURE ENGINEERING & STANDARDIZATION
# ==========================================
print("2. Engineering Per-Capita and Logarithmic Features...")

# Calculate Budget_i: Per-Capita Security Expenditure
# This controls for population scale differences (e.g., Lagos vs. Bayelsa)
macro_df['Budget_i'] = macro_df['Security_Budget_Allocation'] / macro_df['State_Population']

# Calculate ln_Budget_i: Logarithmic Transformation
# Using np.log1p (log(1+x)) to handle any true zero values safely and reduce positive skewness
macro_df['ln_Budget_i'] = np.log1p(macro_df['Budget_i'])

# ==========================================
# 3. MERGE WITH CSTI MICRODATA
# ==========================================
print("3. Merging Macro-Fiscal Data with CSTI Survey Index...")

# Load the previously processed Afrobarometer trust scores
try:
    csti_df = pd.read_csv("../data/processed/nigeria_state_csti_index.csv")
    
    # Perform an inner merge on the 'State' column
    final_regression_df = pd.merge(csti_df, macro_df, on="State", how="inner")
    
    # ==========================================
    # 4. FINAL EXPORT
    # ==========================================
    final_regression_df.to_csv("../data/processed/csti_master_dataset.csv", index=False)
    print("\nSUCCESS: Final econometric dataset constructed and saved to 'data/processed/csti_master_dataset.csv'")
    
    # Display the final structure
    print("\nPreview of Final Master Dataset:")
    print(final_regression_df[['State', 'CSTI', 'Budget_i', 'ln_Budget_i']].head())

except FileNotFoundError:
    print("Error: 'nigeria_state_csti_index.csv' not found. Please run Notebook 01 first.")

1. Initializing Macro-Economic Data Wrangling Pipeline...
2. Engineering Per-Capita and Logarithmic Features...
3. Merging Macro-Fiscal Data with CSTI Survey Index...
Error: 'nigeria_state_csti_index.csv' not found. Please run Notebook 01 first.
